# Email Sending Template

This notebook provides a simple template for sending emails using Python.

## Imports

In [1]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders
import os
from dotenv import load_dotenv

load_dotenv()

True

## Configuration

Set up your email credentials. For security, use environment variables or a .env file.

In [2]:
# Email configuration
SMTP_SERVER = "smtp.gmail.com"  # For Gmail
SMTP_PORT = 587  # TLS port

# Get credentials from environment variables
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")  # Use app password for Gmail

# Verify credentials are loaded
if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
    print("Warning: EMAIL_ADDRESS and EMAIL_PASSWORD must be set in your .env file")
else:
    print(f"Email configured for: {EMAIL_ADDRESS}")

Email configured for: marcus.pythontest@gmail.com


## HTML Email

Send an email with HTML formatting.

In [ ]:
def send_html_email(to_email, subject, html_body):
    """Send an email with HTML content."""
    try:
        # Create message
        msg = MIMEMultipart('alternative')
        msg['Subject'] = subject
        msg['From'] = EMAIL_ADDRESS
        msg['To'] = to_email
        
        # Add HTML content
        html_part = MIMEText(html_body, 'html')
        msg.attach(html_part)
        
        # Connect and send
        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
            server.starttls()
            server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
            server.send_message(msg)
        
        print(f"✓ HTML email sent successfully to {to_email}")
        return True
    except Exception as e:
        print(f"✗ Failed to send email: {e}")
        return False

# Example usage (uncomment when ready)
# send_html_email(
#     to_email="recipient@example.com",
#     subject="HTML Test Email",
#     html_body="<html><body><h1>Hello!</h1></body></html>"
# )

## Weekly grocery highlights grid

Build a 4x5 HTML grid from `data/csv/mail_groceries.csv` and use it as the email body.

In [ ]:
import pandas as pd
import datetime

CSV_PATH = "../data/csv/mail_groceries.csv"


def build_grocery_grid_html(csv_path=CSV_PATH, limit=21):
    """Return an HTML string with a 2-column grid (up to 21 items) for email."""
    df = pd.read_csv(csv_path)
    df = df.fillna("")
    df = df.head(limit)

    def card_html(row):
        img = row.get("public_urls", "") or row.get("image_url", "")
        name = row.get("product_name") or row.get("translated_product") or "Item"
        brand = row.get("brand", "")
        price = row.get("price", "")
        store = row.get("store_name", "")
        quantity = row.get("quantity", "")
        unit_type = row.get("unit_type", "")
        units = row.get("units", "")

        price_display = f"{float(price):.2f},-" if price != "" else ""
        qty = f"{quantity} {unit_type}".strip() if quantity != "" else ""
        unit = f"{units}x {qty}" if units not in ["", None] else qty

        optional_brand = f"<div style='font-size:12px;color:#666;margin-top:4px;'>{brand}</div>" if brand else ""
        optional_unit = f"<div style='font-size:12px;color:#666;'>{unit}</div>" if unit else ""
        img_tag = (
            f"<img src='{img}' alt='{name}' style='width:100%;height:180px;object-fit:cover;border-radius:8px 8px 0 0;'/>"
            if img
            else "<div style='height:180px;background:#eee;border-radius:8px 8px 0 0;'></div>"
        )

        return f"""
                <div style='border:1px solid #e5e5e5;border-radius:10px;overflow:hidden;font-family:Arial, sans-serif;background:#fff;'>
                  {img_tag}
                  <div style='padding:10px;'>
                    <div style='font-weight:600;font-size:14px;color:#222;'>{name}</div>
                    {optional_brand}
                    <div style='font-size:13px;color:#111;margin-top:6px;'>{price_display}</div>
                    <div style='font-size:12px;color:#666;'>{store}</div>
                    {optional_unit}
                  </div>
                </div>
                """

    cards = [card_html(row) for _, row in df.iterrows()]

    rows = []
    for i in range(0, len(cards), 2):
        chunk = cards[i : i + 2]
        cells = "".join(
            f"<td style='padding:8px;vertical-align:top;width:50%;'>" + card + "</td>" for card in chunk
        )
        rows.append(f"<tr>{cells}</tr>")

    table_rows = "\n".join(rows)
    html = f"""
            <html>
              <body style='margin:0;padding:16px;background:#f7f7f7;font-family:Arial,sans-serif;'>
                <div style='max-width:960px;margin:0 auto;'>
                  <p style='color:#444;margin-top:0;margin-bottom:16px;'>Curated picks from this week's sales flyer based on your preferences.</p>
                  <table role='presentation' cellpadding='0' cellspacing='0' border='0' style='border-collapse:collapse;width:100%;table-layout:fixed;'>
                    {table_rows}
                  </table>
                </div>
              </body>
            </html>
            """
    return html


# Build HTML for top 21 items (2-column grid)
try:
    html_content = build_grocery_grid_html()
    preview = html_content[:500] + ("..." if len(html_content) > 500 else "")
    print("HTML preview (first 500 chars):")
    print(preview)
except FileNotFoundError:
    html_content = ""
    print("CSV not found at ../data/csv/mail_groceries.csv")

subject_line = f"Weekly Grocery Highlights — {datetime.date.today().strftime('%Y-%m-%d')}"

# Example send (uncomment when ready)
send_html_email(
    to_email="marcus.presutti.eu@gmail.com",
    subject=subject_line,
    html_body=html_content
)

HTML preview (first 500 chars):

<html>
  <body style='margin:0;padding:16px;background:#f7f7f7;font-family:Arial,sans-serif;'>
    <div style='max-width:960px;margin:0 auto;'>
      <p style='color:#444;margin-top:0;margin-bottom:16px;'>Curated picks from this week's sales flyer based on your preferences.</p>
      <table role='presentation' cellpadding='0' cellspacing='0' border='0' style='border-collapse:collapse;width:100%;table-layout:fixed;'>
        <tr><td style='padding:8px;vertical-align:top;width:50%;'>
<div style='...
✓ HTML email sent successfully to marcus.presutti.eu@gmail.com
✓ HTML email sent successfully to marcus.presutti.eu@gmail.com


True